# Compute Heat-Stress Index Degree

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# import the csv data from meteorology
data_10_14 = pd.read_csv('Athalassa_2010_2014.csv',low_memory = False)
data_15_24 = pd.read_csv('Athalassa_2015_2024.csv',low_memory = False)
# sort the data 
data_10_14 = data_10_14.sort_values(by = 'DateTime') 
data_15_24 = data_15_24.sort_values(by = 'DateTime') 

In [ ]:
# remove duplicate measurements that are measured with Babuc method (a datetime with two different measurements) 2010 - 2014
mask = data_10_14.duplicated(subset = 'DateTime', keep = False) & data_10_14['Type'].str.contains('Babuc', case = False, na = False)
data_10_14 = data_10_14[~mask]

In [ ]:
# Now check if I stiil have duplicates
duplicate_dates = data_10_14['DateTime'][data_10_14['DateTime'].duplicated()].unique()
print('Duplicate DateTime values:', duplicate_dates)

In [ ]:
data_10_14.rename(columns={'AIR TEMPERATURE 1.2m(av) - (°C)': 'Tav'}, inplace=True)      # rename the column of AVERAGE Temperature (for simplicitly)
data_10_14.rename(columns={'RELATIVE HUMIDITY 1.2m(av) - (%)': 'RHav'}, inplace=True)    # rename the column of AVERAGE Humidity (for simplicitly)
#
data_15_24.rename(columns={'AIR TEMPERATURE 1.2m(max) - (°C)': 'Tmax'}, inplace=True)    # rename the column of MAX Temperature (for simplicitly)
data_15_24.rename(columns={'AIR TEMPERATURE 1.2m(min) - (°C)': 'Tmin'}, inplace=True)    # rename the column of MIN Temperature (for simplicitly)
data_15_24.rename(columns={'RELATIVE HUMIDITY 1.2m(max) - (%)': 'RHmax'}, inplace=True)  # rename the column of MAX Humidity (for simplicitly)
data_15_24.rename(columns={'RELATIVE HUMIDITY 1.2m(min) - (%)': 'RHmin'}, inplace=True)  # rename the column of MIN Humidity (for simplicitly)

In [ ]:
# Replace Nan Temperature and Humidity values with the previous row's values
data_10_14['Tav'] = data_10_14['Tav'].ffill()                # data 2010-2014
data_10_14['RHav'] = data_10_14['RHav'].ffill()
#   
data_15_24['Tmax'] = data_15_24['Tmax'].ffill()              # data 2015 - 2024
data_15_24['Tmin'] = data_15_24['Tmin'].ffill()
data_15_24['RHmax'] = data_15_24['RHmax'].ffill()  
data_15_24['RHmin'] = data_15_24['RHmin'].ffill()

In [ ]:
# Verify that all NAN values have been removed 
is_nan = data_15_24['RHmin'].isna().any()            # for Tav and RHav data 2010-2014
print(is_nan)                                        # and Tmax, Tmin, RHmax, RHmin

In [ ]:
# Remove outliers from the data
data_15_24 = data_15_24[data_15_24['RHmax'] <= 100]    # Delete rows where MAX relative humidity has values greater than 100
data_15_24 = data_15_24[data_15_24['RHmin'] <= 100]    # Delete rows where MIN relative humidity has values greater than 100
data_15_24 = data_15_24[data_15_24['Tmax'] <= 50]      # Delete rows where MAX temperature has values greater than 50
data_15_24 = data_15_24[data_15_24['Tmin'] <= 50]      # Delete rows where MIN temperature has values greater than 503
#
data_10_14 = data_10_14[data_10_14['RHav'] <= 100]     # Delete rows where AVERAGE relative humidity has values greater than 100
data_10_14 = data_10_14[data_10_14['Tav'] <= 50]       # Delete rows where AVERAGE temperature has values greater than 50

In [ ]:
# For the data 2015 - 2024: Find the average value of temperature and humidity for each row (10 min)
data_15_24['AvTemp'] = data_15_24[['Tmax', 'Tmin']].mean(axis=1)
data_15_24['AvHum'] = data_15_24[['RHmax', 'RHmin']].mean(axis=1)

In [ ]:
# Calculate the Heat Stress Index 2015 - 2024
data_15_24['THI'] = 1.8*data_15_24['AvTemp'] + 32 - (0.55-0.0055*data_15_24['AvHum']) * (1.8*data_15_24['AvTemp'] + 32 - 58)
# Calculate the Heat Stress Index 2010 - 2014
data_10_14['THI'] = 1.8*data_10_14['Tav'] + 32 - (0.55-0.0055*data_10_14['RHav']) * (1.8*data_10_14['Tav'] + 32 - 58)

In [ ]:
# Combine the two data from 2010 to 2014 and 2015 to 2024 
#
# Convert the groupby results into DataFrames and reset the index
df_15_24 = data_15_24.reset_index()
df_10_14 = data_10_14.reset_index()
#
# Concatenate the two DataFrames vertically
THI_per_day = pd.concat([df_15_24[['DateTime', 'THI']], 
                         df_10_14[['DateTime', 'THI']]])
#
# Reset index after concatenation
THI_per_day = THI_per_day.sort_values(by='DateTime').reset_index(drop=True)
#
# Print the merged data
print(THI_per_day)

In [ ]:
# Count degree hours with THI>72 we have per day
#
# Convert DateTime column to datetime if not already
THI_per_day['DateTime'] = pd.to_datetime(THI_per_day['DateTime'], format='mixed', dayfirst=True)
#
# Calculate the THI excess above 72
THI_per_day['THI_above_72'] = THI_per_day['THI'] - 72
THI_per_day['THI_above_72'] = THI_per_day['THI_above_72'].apply(lambda x: x if x > 0 else 0)
# Floor to the hour using lowercase 'h'
THI_per_day['Hour'] = THI_per_day['DateTime'].dt.floor('h')
# Sum THI degrees above 72 per hour
hourly_excess = THI_per_day.groupby('Hour')['THI_above_72'].sum().reset_index(name='THI_excess_sum_per_hour')
# Count number of 10-min intervals per hour
intervals_per_hour = THI_per_day.groupby('Hour').size().reset_index(name='Intervals_per_hour')
# Merge to get both in one dataframe
hourly_stats = pd.merge(hourly_excess, intervals_per_hour, on='Hour', how='left')
# Calculate average excess per interval
hourly_stats['Avg_excess_per_interval'] = hourly_stats['THI_excess_sum_per_hour'] / hourly_stats['Intervals_per_hour']
# Round for readability
hourly_stats = hourly_stats.round({'Avg_excess_per_interval': 4})
#
# Count the daily THI degree 
hourly_stats['DateTime'] = hourly_stats['Hour'].dt.date
daily_proportion = hourly_stats.groupby('DateTime')['Avg_excess_per_interval'].sum().reset_index(name='Daily THI Degree')
# Show result
print(daily_proportion)

In [ ]:
# Plot the results
plt.figure(figsize=(14, 6))
plt.scatter(daily_proportion['DateTime'], daily_proportion['Daily THI Degree'], color='blue', s=10)
plt.title('Daily THI Degree (for THI > 72)', fontsize=14)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Daily THI Degree', fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.show()